## LangGraphの基本

In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [3]:
import sys
!{sys.executable} -m pip install langgraph


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages

# ステートの定義
class State(TypedDict):
    # データを保存する属性
    messages: Annotated[list, add_messages]

# ステートグラフの作成
graph_builder = StateGraph(State)

In [3]:
# 言語モデルの定義
llm = ChatOpenAI(model_name=MODEL_NAME)

# チャットボットノードの作成
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

# グラフにチャットボットノードを追加
graph_builder.add_node("chatbot", chatbot)

# 開始ノードの指定
graph_builder.set_entry_point("chatbot")
# 終了ノードの指定
graph_builder.set_finish_point("chatbot")

# 実行可能なステートグラフの作成
graph = graph_builder.compile()

In [4]:
# グラフの実行
response = graph.invoke({"messages": [("user", "光の三原色は？")]})

# 結果の表示
print(response)

{'messages': [HumanMessage(content='光の三原色は？', additional_kwargs={}, response_metadata={}, id='3af96eda-2c14-4823-ba5f-43441b5f32e4'), AIMessage(content='光の三原色は、赤（Red）、緑（Green）、青（Blue）の3色です。これらの色を組み合わせることで、さまざまな色を作り出すことができます。この原理は、RGBカラーシステムに基づいており、ディスプレイやテレビなどのデジタルデバイスで広く使用されています。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 14, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DD4wTRhwUPmSp5e20yVZUsXQ12Ywg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c9401-35f2-77b0-aac8-5ae4c763de3e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8

In [5]:
# 言語モデルからの回答のみを表示
print(response["messages"][-1].content)

光の三原色は、赤（Red）、緑（Green）、青（Blue）の3色です。これらの色を組み合わせることで、さまざまな色を作り出すことができます。この原理は、RGBカラーシステムに基づいており、ディスプレイやテレビなどのデジタルデバイスで広く使用されています。


In [7]:
# グラフの実行と結果の表示
def stream_graph_updates(user_input: str):
    # 結果をストリーミングで得る
    events = graph.stream({"messages": [("user", user_input)]})
    for event in events:
        for value in event.values():
            print("回答:", value["messages"][-1].content, flush=True)

# チャットボットのループ
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    print("質問:", user_input, flush=True)
    stream_graph_updates(user_input)

質問: こんにちは！
回答: こんにちは！どうされましたか？何かお手伝いできることがあれば教えてください。
質問: aで始まる英単語を5つ教えて
回答: もちろんです！以下は「a」で始まる英単語の例です：

1. apple（リンゴ）
2. airplane（飛行機）
3. animal（動物）
4. art（芸術）
5. adventure（冒険）

何か他にお手伝いできることがあれば教えてください！
質問: 3つ目の英単語は何ですか？
回答: 「3つ目の英単語」というのが具体的に何を指しているのか不明ですが、もう少し詳細を教えていただければ、必要な情報を提供できると思います。たとえば、特定の文章やテーマに関連する英単語が必要ですか？それとも、一般的な英単語のリストから3つ目の単語が知りたいですか？詳細を教えてください。
ありがとうございました!


In [8]:
# 記憶を持たせる
from langgraph.checkpoint.memory import MemorySaver

# チェックポインタの作成
memory = MemorySaver()

# 記憶を持つ実行可能なステートグラフの作成
memory_graph = graph_builder.compile(checkpointer=memory)

In [9]:
# グラフの実行と結果の表示
def stream_graph_updates(user_input: str):
    events = memory_graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values")
    # 結果をストリーミングで得る
    for event in events:
        print(event["messages"][-1].content, flush=True)

# チャットボットのループ
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    stream_graph_updates(user_input)

こんにちは！
こんにちは！何かお手伝いできることがありますか？
aで始まる英単語を5つ教えて
もちろんです！以下に「a」で始まる英単語を5つ挙げます：

1. Apple（リンゴ）
2. Ant（蟻）
3. Amazing（驚くべき）
4. Art（芸術）
5. Adventure（冒険）

他に知りたい単語やテーマがあれば教えてください！
4つ目の英単語は何ですか？
4つ目の英単語は「Art」です。これは「芸術」や「アート」を意味します。更に詳しい説明や、他の単語について知りたいことがあればお知らせください！
5つ目の英単語は何ですか？
5つ目の英単語は「Adventure」です。これは「冒険」を意味します。新しい経験や挑戦を指す言葉です。他に知りたいことがあれば教えてください！
ありがとうございました!
